# Collecting Heating Data
by Tom Bignell
September 2025

A project to see what data we can get about heating a home and what insights we can get from this data. 

In [75]:
import requests
import webbrowser
import time
import json
import os
from dotenv import load_dotenv
import pandas as pd

## Get Access

Our heating is controlled by a Tado system. Therefore we need to start by getting access to the Tado system. 
They have an API by which we can gain access. 

To start, save your Client ID in a .env file in the same location as this code. 

The cell below uses this Client ID to gain an acess token by which you can access the Tado system. You also need to input your username and password into the browser when it opens. 

The script performs a secure OAuth 2.0 device authorization flow with the Tado API, allowing a user to authenticate without entering credentials directly into the script. Once authorized, it retrieves user account information and stores access tokens locally for future use.

In [76]:
CLIENT_ID = os.getenv("TADO_CLIENT_ID")
DEVICE_AUTH_URL = "https://login.tado.com/oauth2/device_authorize"
TOKEN_URL = "https://login.tado.com/oauth2/token"
API_URL = "https://my.tado.com/api/v2/me"

# Step 1: Request device code
print("🔐 Requesting device code...")
response = requests.post(
    DEVICE_AUTH_URL,
    params={
        "client_id": CLIENT_ID,
        "scope": "offline_access"
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)
data = response.json()
device_code = data["device_code"]
user_code = data["user_code"]
verification_uri = data["verification_uri_complete"]

print(f"📲 Please authorize access by visiting:\n{verification_uri}")
webbrowser.open(verification_uri)

# Step 2: Poll for access token
print("⏳ Waiting for user authorization...")
while True:
    token_response = requests.post(
        TOKEN_URL,
        data={
            "client_id": CLIENT_ID,
            "grant_type": "urn:ietf:params:oauth:grant-type:device_code",
            "device_code": device_code
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"}
    )
    token_data = token_response.json()

    if "access_token" in token_data:
        access_token = token_data["access_token"]
        refresh_token = token_data["refresh_token"]
        print("✅ Access granted!")
        break
    elif token_data.get("error") == "authorization_pending":
        time.sleep(5)
    else:
        print(f"❌ Error: {token_data}")
        exit()

# Step 3: Call Tado API
print("📡 Fetching user info...")
api_response = requests.get(
    API_URL,
    headers={"Authorization": f"Bearer {access_token}"}
)

# Optional: Save refresh token
with open("tado_tokens.json", "w") as f:
    import json
    json.dump({
        "access_token": access_token,
        "refresh_token": refresh_token
    }, f)
    print("💾 Tokens saved to tado_tokens.json")


🔐 Requesting device code...
📲 Please authorize access by visiting:
https://login.tado.com/oauth2/device?user_code=RYDS8D
⏳ Waiting for user authorization...
✅ Access granted!
📡 Fetching user info...
💾 Tokens saved to tado_tokens.json


## Get Home Id

Tado gives each home an id, I guess a user could have more than one home. 

The code below gets the first home id and name.

If a user has multiple houses we would have to introduce the code here to chose which house to run the code for.

In [ ]:
# Load saved tokens
with open("tado_tokens.json", "r") as f:
    tokens = json.load(f)

access_token = tokens["access_token"]

# Request user information
response = requests.get(
    "https://my.tado.com/api/v2/me",
    headers={"Authorization": f"Bearer {access_token}"}
)

# Save home id
home_id = json.loads(response.content).get('homes')[0].get('id')
home_name = json.loads(response.content).get('homes')[0].get('name')

print(f"💾 Home ID saved for:", home_name)

💾 Home ID saved for: Sylvia Avenue


## Get Room ID

Tado then assigns the different room heating setups and hot water different options.

In this cas we have one room and hot water, so Room 1 is 2 and Hot Water is 0.

In [78]:
# Request home information
zones_url = f"https://my.tado.com/api/v2/homes/{home_id}/zones"
zones_response = requests.get(zones_url, headers={"Authorization": f"Bearer {access_token}"})

# Save room and hot water id
room_id = zones_response.json()[0].get('id')
hotwater_id = zones_response.json()[1].get('id')

print(f"💾 Room and Hot Water ID saved for:", home_name)

💾 Room and Hot Water ID saved for: Sylvia Avenue


## Get the Data

We can then select a date to get the data for. From Tado we can collect data for:
- internal temperature
- internal humidity
- target heating temperature
- heating
- hot water
- weather

In [79]:
# Select a date to get the data for
date_str = "2025-01-01"

# Request room information for that date
room_url = f"https://my.tado.com/api/v2/homes/{home_id}/zones/{room_id}/dayReport?date={date_str}"
room_response = requests.get(room_url, headers={"Authorization": f"Bearer {access_token}"})

# Save the data as dictionaries
internal_temp = room_response.json().get('measuredData').get('insideTemperature').get('dataPoints')
humidity = room_response.json().get('measuredData').get('humidity').get('dataPoints')
target_temp = room_response.json().get('stripes').get('dataIntervals')
heating = room_response.json().get('callForHeat').get('dataIntervals')
hot_water = room_response.json().get('hotWaterProduction').get('dataIntervals')
weather = room_response.json().get('weather').get('dataIntervals')

## Temperature Data
- Put the dictionary data into a datafame. 
- Turn the time into a datetime
- Extract the celcius value from the temperature values

In [80]:
#Turn dataPoints into a dataframe
df_internal_temp = pd.DataFrame(internal_temp)

#Convert the date and time to a date and time value
df_internal_temp['timestamp'] = pd.to_datetime(df_internal_temp['timestamp'])

#The value column contains both celsius and farenheit values, split this out
df_internal_temp = pd.concat([df_internal_temp.drop(["value"],axis=1), df_internal_temp["value"].apply(pd.Series)], axis=1)

#Remove the fahrenheit column
df_internal_temp = df_internal_temp.drop(["fahrenheit"], axis = 1)

#Rename the celsius column
df_internal_temp = df_internal_temp.rename(columns={"celsius":"internal_temperature_celsius"})

df_internal_temp

,timestamp,internal_temperature_celsius
0,2024-12-31 23:45:00+00:00,17.43
1,2025-01-01 00:00:00+00:00,17.37
2,2025-01-01 00:15:00+00:00,17.30
3,2025-01-01 00:30:00+00:00,17.30
4,2025-01-01 00:45:00+00:00,17.26
...,...,...
94,2025-01-01 23:15:00+00:00,17.74
95,2025-01-01 23:30:00+00:00,17.65
96,2025-01-01 23:45:00+00:00,17.65
97,2025-01-02 00:00:00+00:00,17.52


In [81]:
#Turn dataPoints into a dataframe
df_humidity = pd.DataFrame(humidity)

#Convert the date and time to a date and time value
df_humidity['timestamp'] = pd.to_datetime(df_humidity['timestamp'])

#Rename the celsius column
df_humidity = df_humidity.rename(columns={"value":"humidity_percentage"})

df_humidity

,timestamp,humidity_percentage
0,2024-12-31 23:45:00+00:00,0.695
1,2025-01-01 00:00:00+00:00,0.695
2,2025-01-01 00:15:00+00:00,0.695
3,2025-01-01 00:30:00+00:00,0.695
4,2025-01-01 00:45:00+00:00,0.694
...,...,...
94,2025-01-01 23:15:00+00:00,0.714
95,2025-01-01 23:30:00+00:00,0.717
96,2025-01-01 23:45:00+00:00,0.717
97,2025-01-02 00:00:00+00:00,0.713
